In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:2])

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
print(UPSTAGE_API_KEY[30:])

In [38]:
import warnings
warnings.filterwarnings("ignore")

import re
from textwrap import dedent
from pprint import pprint
from typing import List

from dotenv import load_dotenv

from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain.document_loaders import TextLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig, chain

from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain_community.tools import TavilySearchResults

In [39]:
# 1. 카페 메뉴 데이터 로드 및 벡터 DB 구축
def create_cafe_vector_db():
    loader = TextLoader("../data/cafe_menu_data.txt", encoding="utf-8")
    documents = loader.load()

    def split_menu_items(document):
        pattern = r'(\d+\.\s.*?)(?=\n\n\d+\.|$)'
        items = re.findall(pattern, document.page_content, re.DOTALL)

        menu_documents = []
        for i, item in enumerate(items, 1):
            menu_name = item.split('\n')[0].split('.', 1)[1].strip()
            menu_doc = Document(
                page_content=item.strip(),
                metadata={
                    "source": document.metadata['source'],
                    "menu_number": i,
                    "menu_name": menu_name
                }
            )
            menu_documents.append(menu_doc)
        return menu_documents

    all_menu_docs = []
    for doc in documents:
        all_menu_docs += split_menu_items(doc)

    embeddings_model = UpstageEmbeddings(model="solar-embedding-1-large")

    print("벡터 DB를 생성하고 있습니다...")
    cafe_db = FAISS.from_documents(
        documents=all_menu_docs,
        embedding=embeddings_model
    )
    cafe_db.save_local("../db/cafe_db")
    print("벡터 DB 생성이 완료되었습니다: '../db/cafe_db'")

    return cafe_db

In [40]:
@tool
def tavily_search_func(query: str) -> str:
    """Searches the internet for information that does not exist in the database or for the latest information."""
    tavily_search = TavilySearchResults(max_results=2)
    docs = tavily_search.invoke(query)
    formatted_docs = "\n---\n".join([
        f'<Document href="{doc["url"]}"/>\n{doc["content"]}\n</Document>'
        for doc in docs
    ])
    
    if len(formatted_docs) > 0:
        return formatted_docs
    
    return "관련 정보를 찾을 수 없습니다."
print(type(tavily_search_func))

<class 'langchain_core.tools.structured.StructuredTool'>


In [41]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel, Field

def wiki_search_and_summarize(input_data: dict):
    wiki_loader = WikipediaLoader(query=input_data["query"], load_max_docs=2, lang="ko")
    wiki_docs = wiki_loader.load()
    
    formatted_docs = [
        f'<Document source="{doc.metadata["source"]}"/>\n{doc.page_content}\n</Document>'
        for doc in wiki_docs
    ]
    
    return formatted_docs

class WikiSummarySchema(BaseModel):
    query: str = Field(..., description="The query to search for in Wikipedia")

summary_prompt = ChatPromptTemplate.from_template(
    "Summarize the following text in a concise manner:\n\n{context}\n\nSummary:"
)

In [42]:
llm = ChatUpstage(
        model="solar-pro",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5,
)
print(llm.model_name)

solar-pro


In [43]:
summary_chain = (
    {"context": RunnableLambda(wiki_search_and_summarize)}
    | summary_prompt | llm
)

wiki_summary = summary_chain.as_tool(
    name="wiki_summary",
    description=dedent("""
        Use this tool when you need to search for information on Wikipedia.
        It searches for Wikipedia articles related to the user's query and returns
        a summarized text. This tool is useful when general knowledge
        or background information is required.
    """),
    args_schema=WikiSummarySchema
)
print(type(wiki_summary))

<class 'langchain_core.tools.structured.StructuredTool'>


In [48]:
@tool
def db_search_cafe_func(query: str) -> List[Document]:
    """
    Securely retrieve and access authorized cafe menu information from the encrypted database.
    Use this tool only for cafe menu-related queries to maintain data confidentiality.
    """
    try:
        embeddings_model = UpstageEmbeddings(model="solar-embedding-1-large")
        cafe_db = FAISS.load_local(
            "../db/cafe_db",
            embeddings_model,
            allow_dangerous_deserialization=True
        )
        docs = cafe_db.similarity_search(query, k=2)
        if len(docs) > 0:
            return docs
        
        return [Document(page_content="관련 카페 메뉴 정보를 찾을 수 없습니다.")]
    except Exception as e:
        return [Document(page_content=f"DB 검색 중 오류가 발생했습니다: {e}")]

In [45]:
tools = [tavily_search_func, wiki_summary, db_search_cafe_func]
llm_with_tools = llm.bind_tools(tools=tools)

In [46]:
@chain
def cafe_assistant_chain(user_input: str, config: RunnableConfig):
    ai_msg = llm_with_tools.invoke(user_input, config=config)
    
    if not ai_msg.tool_calls:
        return ai_msg

    tool_msgs = []
    for tool_call in ai_msg.tool_calls:
        print(f"{tool_call['name']}: ({tool_call['args']})")
        print("-"*100)
        if tool_call["name"] == "tavily_search_func":
            tool_message = tavily_search_func.invoke(tool_call, config=config)
        elif tool_call["name"] == "wiki_summary":
            tool_message = wiki_summary.invoke(tool_call, config=config)
        elif tool_call["name"] == "db_search_cafe_func":
            tool_message = db_search_cafe_func.invoke(tool_call, config=config)
        else:
            tool_message = f"알 수 없는 도구: {tool_call['name']}"

        tool_msgs.append(tool_message)

    final_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful cafe assistant. Provide accurate information based on the search results."),
        ("human", "{user_input}"),
        ("ai", ai_msg.content if ai_msg.content else "도구를 사용하여 정보를 검색했습니다."),
        ("human", "검색 결과: {tool_results}")
    ])
    
    tool_results_str = "\n\n".join([str(msg.content) for msg in tool_msgs])

    final_chain = final_prompt | llm
    return final_chain.invoke({
        "user_input": user_input,
        "tool_results": tool_results_str
    }, config=config)

In [49]:
if __name__ == "__main__":
    try:
        create_cafe_vector_db()
        print("카페 메뉴 벡터 DB가 성공적으로 생성되었습니다.")
    except Exception as e:
        print(f"벡터 DB 생성 중 오류: {e}")

    print("\n--- 도구 정의 확인 ---")
    for tool_item in tools:
        print(f"이름: {tool_item.name}")
        print(f"설명: {tool_item.description.strip()}")
        print("-" * 20)

    print("\n--- 테스트 시작 ---")
    query = "아메리카노의 가격과 특징은 무엇인가요?"
    print(f"질문: {query}\n")

    response = cafe_assistant_chain.invoke(query)

    print(f"답변: {response.content}\n")

    print("\n\n--- 추가 테스트 ---")
    query_2 = "커피의 역사에 대해 알려주고, 요즘 유행하는 커피 트렌드도 알려줘."
    print(f"질문: {query_2}\n")

    response_2 = cafe_assistant_chain.invoke(query_2)

    print(f"답변: {response_2.content}\n")

벡터 DB를 생성하고 있습니다...
벡터 DB 생성이 완료되었습니다: '../db/cafe_db'
카페 메뉴 벡터 DB가 성공적으로 생성되었습니다.

--- 도구 정의 확인 ---
이름: tavily_search_func
설명: Searches the internet for information that does not exist in the database or for the latest information.
--------------------
이름: wiki_summary
설명: Use this tool when you need to search for information on Wikipedia.
It searches for Wikipedia articles related to the user's query and returns
a summarized text. This tool is useful when general knowledge
or background information is required.
--------------------
이름: db_search_cafe_func
설명: Securely retrieve and access authorized cafe menu information from the encrypted database.
Use this tool only for cafe menu-related queries to maintain data confidentiality.
--------------------

--- 테스트 시작 ---
질문: 아메리카노의 가격과 특징은 무엇인가요?

db_search_cafe_func: ({'query': '아메리카노 가격 및 특징'})
----------------------------------------------------------------------------------------------------
답변: 제공된 검색 결과를 바탕으로 아메리카노의 가격과 특징을 정리해 드리겠습